# CSE457 – Lab 5: Speech Enhancement, Speaker Recognition và Speaker Separation

- **Họ tên:** Phạm Duy Long  **MSSV:** 2251061829  **Lớp:** 64CNTT1
- **Case study "Trợ lý phòng họp mini":** (1) làm sạch câu nói bị nhiễu quạt (spectral subtraction / Wiener trong miền STFT); (2) đoán người nói trong 3 người đã enrollment (MFCC thống kê + cosine), kèm verification FAR/FRR; (3) hai người nói chồng lấn → hai mixture microphone → tách bằng FastICA.
- **Cách chạy:** đặt dữ liệu vào `lab5_data/` (xem README; thu bằng `record_dataset.py`), *Kernel → Restart & Run All*. Mọi tham số nằm trong cell `CFG` bên dưới.

Mục lục: 4.1 tạo noisy speech → 4.2 spectral subtraction → 4.3 Wiener + so sánh (TN1–TN3) → 4.4 speaker identification (TN4–TN6) → 4.5 verification → 4.6–4.7 mixture + FastICA (TN7) → 4.8 tích hợp (TN8, câu 12) → trả lời câu hỏi.

In [ ]:
import re, io, time, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
from numpy.linalg import norm
from scipy.signal import correlate
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import FastICA
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

warnings.filterwarnings("ignore")
FS, WIN, HOP, NFFT = 16000, 400, 160, 512          # 25 ms / 10 ms / NFFT 512 (mục 3.3)

DATA = Path("lab5_data"); CLEAN, NOISE, SEP = DATA / "clean", DATA / "noise", DATA / "separation"
RES = Path("results"); AUD, FIG = RES / "audio", RES / "figures"
for p in (AUD, FIG):
    p.mkdir(parents=True, exist_ok=True)

# ---- toàn bộ tham số thí nghiệm ở MỘT chỗ ----
CFG = dict(
    seed=0,
    snrs=[0, 5, 10],                       # mức SNR đầu vào (dB)
    beta=1.5, gamma=0.02,                  # tham số mặc định của spectral subtraction (β, floor γ)
    beta_grid=[1.0, 1.5, 2.0], gamma_grid=[0.0, 0.01, 0.05],
    trim_speaker=True, trim_db=30,         # cắt silence đầu/cuối file người nói
    n_mfcc=20,                             # MFCC cho speaker embedding (gợi ý D = 20)
    token_grid=[1, 2, 4],                  # số enrollment utterance (TN6)
    case_speaker="spk01", case_test_idx=0, case_noise="fan", case_snr=5,   # case study
    A_list={"dễ": [[1.0, 0.6], [0.4, 1.0]], "trung bình": [[1.0, 0.85], [0.8, 1.0]],
            "gần suy biến": [[1.0, 0.97], [0.96, 1.0]]},
    ica_noise_snrs=[30, 20, 10],           # SNR của nhiễu cộng thêm vào hai mixture (TN8)
)
np.random.seed(CFG["seed"])
print("librosa", librosa.__version__, "| numpy", np.__version__)

## Dữ liệu và hàm tiện ích

Cấu trúc: `clean/<spkXX>_{enroll,test}_NN.wav`, `noise/{fan,cafeteria}.wav`, `separation/source_{A,B}.wav`. File người nói được cắt silence đầu/cuối. Enrollment và test nằm ở **file khác nhau**; threshold verification chỉ chọn trên tập dev tách từ enrollment (mục 4.5).

In [ ]:
def load16k(path):
    x, sr = librosa.load(str(path), sr=FS, mono=True)
    return x.astype(np.float64), sr

def power(x):
    return np.mean(np.asarray(x, dtype=np.float64) ** 2) + 1e-12

def rms(x):
    return float(np.sqrt(power(x)))

def add_noise_at_snr(s, v, snr_db_target):
    # y = s + α v với α = sqrt(Ps / (Pv · 10^(R/10)))  (mục 3.2)
    if len(v) < len(s):
        v = np.tile(v, int(np.ceil(len(s) / len(v))))
    v = v[:len(s)]
    alpha = np.sqrt(power(s) / (power(v) * 10 ** (snr_db_target / 10)))
    return s + alpha * v, alpha, v

def snr_db(ref, test):
    L = min(len(ref), len(test)); ref, test = ref[:L], test[:L]
    err = test - ref
    return 10 * np.log10(np.sum(ref ** 2) / (np.sum(err ** 2) + 1e-12))

def snr_aligned(ref, est, max_lag=480):
    # căn chỉnh độ trễ (±30 ms) rồi tính SNR; trả về (SNR, lag mẫu)
    L = min(len(ref), len(est)); ref, est = ref[:L], est[:L]
    c = correlate(est, ref, mode="full", method="fft"); lags = np.arange(-L + 1, L)
    sel = np.abs(lags) <= max_lag
    lag = int(lags[sel][np.argmax(c[sel])])
    if lag > 0:
        est = np.concatenate([est[lag:], np.zeros(lag)])
    elif lag < 0:
        est = np.concatenate([np.zeros(-lag), est[:L + lag]])
    return snr_db(ref, est), lag

def save_wav(path, x):
    x = np.asarray(x, dtype=np.float64)
    pk = np.max(np.abs(x))
    if pk > 0.99:                      # chỉ để tránh clipping khi LƯU file; số đo luôn tính trên bản chưa scale
        x = x * (0.95 / pk)
    sf.write(str(path), x.astype(np.float32), FS, subtype="PCM_16")

def show_spec(ax, x, title="", fmax=8000, vmin=-80):
    S = librosa.amplitude_to_db(np.abs(librosa.stft(x, n_fft=NFFT, hop_length=HOP, win_length=WIN)), ref=1.0, top_db=None)
    S = S - S.max()
    ax.imshow(S, origin="lower", aspect="auto", cmap="magma", vmin=vmin, extent=[0, len(x) / FS, 0, FS / 2000])
    ax.set_ylim(0, fmax / 1000); ax.set_title(title); ax.set_ylabel("kHz")

# ---- nạp dữ liệu người nói ----
if not CLEAN.exists() or not list(CLEAN.glob("*.wav")):
    raise FileNotFoundError("Chưa có dữ liệu. Xem README: thu bằng `python record_dataset.py ...` vào lab5_data/.")
pat = re.compile(r"^(spk\d+)_(enroll|test)_(\d+)\.wav$")
SPK = {}
for f in sorted(CLEAN.glob("*.wav")):
    m = pat.match(f.name)
    if not m:
        continue
    x, _ = load16k(f)
    if CFG["trim_speaker"]:
        x, _ = librosa.effects.trim(x, top_db=CFG["trim_db"])
    SPK.setdefault(m[1], {"enroll": [], "test": []})[m[2]].append(dict(name=f.name, x=x))
speakers = sorted(SPK)
for sp in speakers:
    print(f"{sp}: {len(SPK[sp]['enroll'])} enroll, {len(SPK[sp]['test'])} test, "
          f"độ dài TB {np.mean([len(u['x']) for k in SPK[sp] for u in SPK[sp][k]]) / FS:.1f}s")
assert len(speakers) >= 3, "Cần ít nhất 3 người nói"
assert all(len(SPK[s]["enroll"]) >= 4 and len(SPK[s]["test"]) >= 2 for s in speakers), "Mỗi người cần >=4 enroll và >=2 test"

NOISES = {p.stem: load16k(p)[0] for p in sorted(NOISE.glob("*.wav"))}
assert NOISES, "Chưa có file nhiễu trong lab5_data/noise/"
print("Nhiễu:", {k: f"{len(v)/FS:.1f}s, RMS={rms(v):.4f}" for k, v in NOISES.items()})
CASE_NOISE = CFG["case_noise"] if CFG["case_noise"] in NOISES else next(iter(NOISES))
case_s = SPK[CFG["case_speaker"] if CFG["case_speaker"] in SPK else speakers[0]]["test"][CFG["case_test_idx"]]
print("Case study:", case_s["name"], "+", CASE_NOISE, "@", CFG["case_snr"], "dB")

## 4.1 Tạo noisy speech ở SNR cho trước

Công thức (mục 3.2): $\alpha=\sqrt{P_s/(P_v\cdot10^{R/10})}$, $y=s+\alpha v$. Noise được lặp/cắt cho đủ độ dài; **không peak-normalize** trước khi đo. Kiểm tra: SNR đo lại phải lệch ≤ ±0.2 dB so với mục tiêu — đo cả trên mảng float và sau khi lưu/đọc lại WAV 16-bit (lượng tử hoá).

In [ ]:
def roundtrip_snr(s, y):
    # lưu WAV PCM16 với cùng một hệ số scale cho s và y, đọc lại rồi đo SNR (kiểm tra sai số lượng tử hoá)
    g = 0.95 / max(np.max(np.abs(y)), np.max(np.abs(s)))
    def q(x):
        bio = io.BytesIO(); sf.write(bio, (x * g), FS, format="WAV", subtype="PCM_16"); bio.seek(0)
        return sf.read(bio)[0]
    return snr_db(q(s), q(y))

rows = []
for nz, v in NOISES.items():
    for R in CFG["snrs"]:
        s = case_s["x"]
        y, alpha, vv = add_noise_at_snr(s, v, R)
        m_float, m_pcm = snr_db(s, y), roundtrip_snr(s, y)
        rows.append(dict(noise=nz, SNR_muc_tieu=R, alpha=round(alpha, 4), SNR_do_float=round(m_float, 3) + 0.0,
                         SNR_do_sau_PCM16=round(m_pcm, 3) + 0.0, lech_dB=round(abs(m_pcm - R), 3),
                         dat_le_0p2dB=abs(m_pcm - R) <= 0.2))
chk = pd.DataFrame(rows); chk.to_csv(RES / "4_1_snr_check.csv", index=False)
print(f"P_s = {power(case_s['x']):.5f}")
print(chk.to_string(index=False))

# ví dụ số của đề: Ps=0.04, Pv=0.01, R=10 dB -> α ≈ 0.632
print("\nKiểm tra công thức với ví dụ của đề: α =", round(float(np.sqrt(0.04 / (0.01 * 10 ** (10 / 10)))), 3))

s_case, v_case = case_s["x"], NOISES[CASE_NOISE]
noisy = {R: add_noise_at_snr(s_case, v_case, R)[0] for R in CFG["snrs"]}
save_wav(AUD / "clean_ref.wav", s_case)
for R, y in noisy.items():
    save_wav(AUD / f"noisy_{R}dB.wav", y)
fig, ax = plt.subplots(len(CFG["snrs"]) + 1, 2, figsize=(13, 2.6 * (len(CFG["snrs"]) + 1)))
t = np.arange(len(s_case)) / FS
for r, (nm, sig) in enumerate([("clean", s_case)] + [(f"noisy {R} dB ({CASE_NOISE})", noisy[R]) for R in CFG["snrs"]]):
    ax[r, 0].plot(np.arange(len(sig)) / FS, sig, lw=0.5, color="k"); ax[r, 0].set_title(nm); ax[r, 0].set_ylim(-1.2 * np.max(np.abs(noisy[CFG["snrs"][0]])), 1.2 * np.max(np.abs(noisy[CFG["snrs"][0]])))
    show_spec(ax[r, 1], sig, nm)
ax[-1, 0].set_xlabel("giây"); ax[-1, 1].set_xlabel("giây")
plt.tight_layout(); plt.savefig(FIG / "4_1_clean_vs_noisy.png", dpi=130); plt.show()

## 4.2 Spectral subtraction và 4.3 Wiener filter

Cả hai chạy trong miền STFT (hann, win 25 ms = 400 mẫu, hop 10 ms = 160 mẫu, NFFT 512), **giữ pha của tín hiệu noisy**, chỉ sửa độ lớn, rồi ISTFT/overlap-add:

- Spectral subtraction: $|\hat S|^2=\max(|Y|^2-\beta\hat P_v,\ \gamma\hat P_v)$.
- Wiener: $\hat P_s=\max(P_y-\hat P_v,0)$, $H=\hat P_s/(\hat P_s+\hat P_v+\varepsilon)$, $\hat S=H\cdot Y$.

PSD nhiễu $\hat P_v[k]=\frac1M\sum_m|Y_v[m,k]|^2$ ước lượng từ **noise clip**. Lưu ý quan trọng: noise đã được nhân $\alpha$ khi trộn, nên PSD được tính từ $\alpha v$ (mức nhiễu thực sự có trong tín hiệu), nếu dùng nhiễu chưa nhân $\alpha$ thì trừ sai thang. Với bản ghi thật, thay bằng đoạn noise-only của chính bản ghi (so sánh ở thí nghiệm "PSD từ đầu file" bên dưới).

In [ ]:
def stft(x):
    return librosa.stft(x, n_fft=NFFT, hop_length=HOP, win_length=WIN, window="hann")

def istft(X, length):
    return librosa.istft(X, hop_length=HOP, win_length=WIN, window="hann", length=length)

def noise_psd(noise):
    return np.mean(np.abs(stft(noise)) ** 2, axis=1, keepdims=True)           # (bins, 1)

def noise_psd_head(y, n_frames=20):
    # ước lượng PSD từ n_frames frame đầu (giả sử noise-only), bỏ vài frame biên do hiệu ứng center-padding
    return np.mean(np.abs(stft(y)[:, 2:2 + n_frames]) ** 2, axis=1, keepdims=True)

def spectral_subtraction(y, Pn, beta=1.5, gamma=0.02):
    Y = stft(y); Py = np.abs(Y) ** 2
    Pss = np.maximum(Py - beta * Pn, gamma * Pn)
    return istft(np.sqrt(Pss) * np.exp(1j * np.angle(Y)), len(y))

def wiener_filter(y, Pn, eps=1e-12):
    Y = stft(y); Py = np.abs(Y) ** 2
    Ps = np.maximum(Py - Pn, 0.0)
    H = Ps / (Ps + Pn + eps)
    return istft(H * Y, len(y))

def enhance(y, Pn, method, beta=None, gamma=None):
    beta = CFG["beta"] if beta is None else beta
    gamma = CFG["gamma"] if gamma is None else gamma
    return spectral_subtraction(y, Pn, beta, gamma) if method == "ss" else wiener_filter(y, Pn)

def kurt_power(x):
    # kurtosis của công suất STFT – đo "gai" phổ; tỷ số sau/trước > 1 => musical noise (Uemura et al.)
    P = (np.abs(stft(x)) ** 2).ravel()
    return float(np.mean((P - P.mean()) ** 4) / (P.var() ** 2 + 1e-30))

def eval_one(s, v, snr, method, beta=None, gamma=None, psd="clip", lead_s=0.3):
    # Một lượt: tạo noisy, enhance, đo ΔSNR / giảm nhiễu (NR) / musical noise (KR) / méo tiếng (SD)
    if psd == "head":                                    # noisy có đoạn noise-only dẫn đầu, PSD lấy từ đó
        lead = int(lead_s * FS)
        _, alpha, _ = add_noise_at_snr(s, v, snr)
        vv = np.tile(v, int(np.ceil((len(s) + lead) / len(v))))[:len(s) + lead]
        s_pad = np.concatenate([np.zeros(lead), s])
        y = s_pad + alpha * vv
        Pn = noise_psd_head(y, n_frames=int(lead_s * 100) - 4)
        e = enhance(y, Pn, method, beta, gamma)[lead:]
        v_only = alpha * vv
    else:
        y, alpha, vv = add_noise_at_snr(s, v, snr)
        Pn = noise_psd(alpha * vv)                        # PSD của nhiễu ở đúng mức đã trộn
        e = enhance(y, Pn, method, beta, gamma)
        v_only = alpha * vv
    snr_in = snr_db(s, y if psd != "head" else y[int(lead_s * FS):])
    snr_out, lag = snr_aligned(s, e)
    v_out = enhance(v_only, Pn, method, beta, gamma)
    nr = 10 * np.log10(power(v_only) / power(v_out))
    kr = kurt_power(v_out) / kurt_power(v_only)
    sd = snr_db(s, enhance(s, Pn, method, beta, gamma))   # đưa speech sạch qua bộ lọc: cao = ít méo
    return dict(SNR_in=snr_in, SNR_out=snr_out, dSNR=snr_out - snr_in, NR_dB=nr, KR=kr, SD_dB=sd, lag=lag), e, y

# --- Case study: 5 dB, nhiễu quạt ---
rows = []
for R in CFG["snrs"]:
    for method in ["ss", "wiener"]:
        r, e, y = eval_one(s_case, v_case, R, method)
        rows.append(dict(SNR_in_muc_tieu=R, method="SS" if method == "ss" else "Wiener", SNR_in=round(r["SNR_in"], 2),
                         SNR_out=round(r["SNR_out"], 2), dSNR=round(r["dSNR"], 2), lag_mau=r["lag"]))
        if R == CFG["case_snr"]:
            save_wav(AUD / ("enhanced_ss.wav" if method == "ss" else "enhanced_wiener.wav"), e)
        save_wav(AUD / f"enhanced_{'ss' if method == 'ss' else 'wiener'}_{R}dB.wav", e)
case_tbl = pd.DataFrame(rows); case_tbl.to_csv(RES / "4_2_case_snr.csv", index=False)
print(f"Case study ({case_s['name']} + {CASE_NOISE}), β={CFG['beta']}, γ={CFG['gamma']}:")
print(case_tbl.to_string(index=False))

R = CFG["case_snr"]
_, e_ss, y5 = eval_one(s_case, v_case, R, "ss"); _, e_w, _ = eval_one(s_case, v_case, R, "wiener")
fig, ax = plt.subplots(2, 2, figsize=(13, 6))
for a, (nm, sig) in zip(ax.ravel(), [("clean", s_case), (f"noisy {R} dB", y5), ("spectral subtraction", e_ss), ("Wiener", e_w)]):
    show_spec(a, sig, nm, fmax=8000)
plt.tight_layout(); plt.savefig(FIG / "4_2_spectrograms_enh.png", dpi=130); plt.show()

**Nghe A/B/C (bắt buộc, điền sau khi nghe `results/audio/noisy_5dB.wav`, `enhanced_ss.wav`, `enhanced_wiener.wav`):** residual noise …; musical noise (tiếng "chim hót"/gợn) …; độ rõ phụ âm (s, x, t, k) …; độ tự nhiên …. Số SNR cao hơn chưa chắc nghe tốt hơn: thuật toán có thể làm tiếng kim loại hoặc mất phụ âm.

In [ ]:
# --- TN1: SNR đầu vào x loại nhiễu x phương pháp, trung bình trên mọi test utterance ---
UTT = [(sp, u["name"], u["x"]) for sp in speakers for u in SPK[sp]["test"]]
rows = []
for nz, v in NOISES.items():
    for R in CFG["snrs"]:
        for method in ["ss", "wiener"]:
            rs = [eval_one(s, v, R, method)[0] for _, _, s in UTT]
            d = pd.DataFrame(rs)
            rows.append(dict(noise=nz, SNR_in=R, method="SS" if method == "ss" else "Wiener", n_utt=len(rs),
                             SNR_out=d.SNR_out.mean(), dSNR=d.dSNR.mean(), dSNR_std=d.dSNR.std(),
                             NR_dB=d.NR_dB.mean(), KR_musical=d.KR.mean(), SD_dB=d.SD_dB.mean()))
tn1 = pd.DataFrame(rows); tn1.round(2).to_csv(RES / "TN1_input_snr.csv", index=False)
print("TN1 – trung bình trên tất cả test utterance (SS: β=%.1f, γ=%.2f):" % (CFG["beta"], CFG["gamma"]))
print(tn1.round(2).to_string(index=False))
piv = tn1.pivot_table(index=["noise", "SNR_in"], columns="method", values="dSNR")
piv["Wiener - SS"] = piv["Wiener"] - piv["SS"]
print("\nΔSNR (dB) và chênh lệch Wiener − SS:"); print(piv.round(2).to_string())

fig, ax = plt.subplots(1, len(NOISES), figsize=(6 * len(NOISES), 3.6), sharey=True)
for a, nz in zip(np.atleast_1d(ax), NOISES):
    for method, c in [("SS", "tab:orange"), ("Wiener", "tab:blue")]:
        d = tn1[(tn1.noise == nz) & (tn1.method == method)]
        a.errorbar(d.SNR_in, d.dSNR, yerr=d.dSNR_std, marker="o", label=method, color=c, capsize=3)
    a.set_title(f"ΔSNR theo SNR đầu vào – nhiễu {nz}"); a.set_xlabel("SNR vào (dB)"); a.set_xticks(CFG["snrs"]); a.grid(alpha=0.3)
np.atleast_1d(ax)[0].set_ylabel("ΔSNR (dB)"); np.atleast_1d(ax)[0].legend()
plt.tight_layout(); plt.savefig(FIG / "TN1_dsnr_vs_snr.png", dpi=130); plt.show()

**Vì sao Wiener có thể tốt hơn ở 10 dB nhưng kém rõ ở 0 dB (mục 4.3):** hệ số $\hat P_s=\max(P_y-\hat P_v,0)$ được suy ra từ chính hiệu $P_y-\hat P_v$. Ở 10 dB, vùng tần số có speech chiếm ưu thế ($P_s\gg P_v$) cho $H\to1$ và ước lượng $\hat P_s$ chính xác, nên giữ speech và dập nhiễu mượt. Ở 0 dB, $P_s\approx P_v$ ở nhiều ô thời gian–tần số, hiệu $P_y-\hat P_v$ dao động mạnh quanh 0 (sai số ước lượng PSD nhiễu, nhiễu không dừng thật sự) → $\hat P_s$ nhỏ/không ổn định → $H$ nhỏ làm suy giảm cả speech yếu (phụ âm), còn các ô lẻ có $H$ lớn gây gợn. Xem cột "Wiener − SS" ở trên để đối chiếu với số liệu của bạn (dấu của chênh lệch có thể khác tuỳ loại nhiễu).

In [ ]:
# --- TN2: β của spectral subtraction (γ cố định); TN3: floor γ (β cố định). Nhiễu quạt, SNR = case_snr ---
Rcase = CFG["case_snr"]
def sweep(kind, grid):
    rows = []
    for g in grid:
        kw = dict(beta=g) if kind == "beta" else dict(gamma=g)
        rs = [eval_one(s, v_case, Rcase, "ss", **kw)[0] for _, _, s in UTT]
        d = pd.DataFrame(rs)
        rows.append({kind: g, "dSNR": d.dSNR.mean(), "NR_dB": d.NR_dB.mean(), "KR_musical": d.KR.mean(), "SD_dB": d.SD_dB.mean()})
    return pd.DataFrame(rows)
tn2, tn3 = sweep("beta", CFG["beta_grid"]), sweep("gamma", CFG["gamma_grid"])
tn2.round(3).to_csv(RES / "TN2_beta.csv", index=False); tn3.round(3).to_csv(RES / "TN3_gamma.csv", index=False)
print(f"TN2 – β (γ={CFG['gamma']}), {CASE_NOISE} @ {Rcase} dB:"); print(tn2.round(3).to_string(index=False))
print(f"\nTN3 – γ (β={CFG['beta']}), {CASE_NOISE} @ {Rcase} dB:"); print(tn3.round(3).to_string(index=False))
print("\nKR_musical > 1: nhiều gai phổ (musical noise) hơn nhiễu gốc; SD_dB: SNR của speech SẠCH sau khi qua bộ lọc (càng cao càng ít méo).")

_, a_c, vv_c = add_noise_at_snr(s_case, v_case, Rcase)
Pn_case = noise_psd(a_c * vv_c)
fig, ax = plt.subplots(2, 3, figsize=(15, 6))
for c, b in enumerate(CFG["beta_grid"]):
    show_spec(ax[0, c], enhance(y5, Pn_case, "ss", beta=b), f"SS β={b} (γ={CFG['gamma']})")
for c, g in enumerate(CFG["gamma_grid"]):
    show_spec(ax[1, c], enhance(y5, Pn_case, "ss", gamma=g), f"SS γ={g} (β={CFG['beta']})")
plt.tight_layout(); plt.savefig(FIG / "TN2_TN3_beta_gamma_spectrograms.png", dpi=130); plt.show()
for b in CFG["beta_grid"]:
    _, e, _ = eval_one(s_case, v_case, Rcase, "ss", beta=b); save_wav(AUD / f"enhanced_ss_beta{b}.wav", e)
for g in CFG["gamma_grid"]:
    _, e, _ = eval_one(s_case, v_case, Rcase, "ss", gamma=g); save_wav(AUD / f"enhanced_ss_gamma{g}.wav", e)

In [ ]:
# --- Thêm: ước lượng PSD nhiễu từ noise clip (như đề) so với từ đoạn noise-only đầu file (thực tế hơn) ---
rows = []
for method in ["ss", "wiener"]:
    for psd in ["clip", "head"]:
        d = pd.DataFrame([eval_one(s, v_case, Rcase, method, psd=psd)[0] for _, _, s in UTT])
        rows.append(dict(method="SS" if method == "ss" else "Wiener", PSD_tu=("noise clip riêng" if psd == "clip" else "0.3 s noise-only đầu file"),
                         dSNR=d.dSNR.mean(), NR_dB=d.NR_dB.mean()))
psd_tbl = pd.DataFrame(rows); psd_tbl.round(2).to_csv(RES / "extra_psd_source.csv", index=False)
print(psd_tbl.round(2).to_string(index=False))

**Nhận xét TN1–TN3 (điền sau khi xem bảng và nghe):**
- TN1: ΔSNR tăng/giảm … khi SNR vào tăng; chất lượng nghe …
- TN2: β tăng từ 1.0 → 2.0 làm NR_dB … nhưng SD_dB (méo speech) … và KR_musical …; musical noise rõ nhất ở β = …
- TN3: floor γ lớn làm … (giữ lại nền nhiễu nhỏ che các gai phổ) nhưng …

## 4.4 Speaker enrollment và speaker identification

Embedding thống kê của một utterance (mục 3.8): $e=[\mu(c),\sigma(c),\mu(\Delta c),\sigma(\Delta c)]$ với $c$ = 20 MFCC, rồi **L2-normalize**. Mô hình người nói $m_i$ = trung bình các embedding enrollment (rồi L2-normalize). Nhận dạng: $\hat i=\arg\max_i\cos(e,m_i)$.

Đây là baseline sư phạm, không phải speaker embedding hiện đại. Các biến thể cho ablation (TN4, TN5): **MFCC-only** (bỏ Δ), **CMN** (trừ trung bình MFCC theo utterance), và thêm **z-score theo thống kê enrollment** (tuỳ chọn mở rộng, chỉ dùng enrollment để tính).

In [ ]:
def raw_embed(x, delta=True, cmn=False):
    M = librosa.feature.mfcc(y=x, sr=FS, n_mfcc=CFG["n_mfcc"], n_fft=NFFT, win_length=WIN, hop_length=HOP, window="hann")
    if cmn:
        M = M - M.mean(axis=1, keepdims=True)
    parts = [M.mean(axis=1), M.std(axis=1)]
    if delta:
        D = librosa.feature.delta(M)
        parts += [D.mean(axis=1), D.std(axis=1)]
    return np.concatenate(parts)

def l2(e):
    return e / (norm(e) + 1e-12)

def cosine(a, b):
    return float(np.dot(a, b) / (norm(a) * norm(b) + 1e-12))

def make_embedder(delta=True, cmn=False, z=False):
    stats = None
    if z:
        R_ = np.stack([raw_embed(u["x"], delta, cmn) for sp in speakers for u in SPK[sp]["enroll"]])
        stats = (R_.mean(axis=0), R_.std(axis=0) + 1e-8)
    def emb(x):
        r = raw_embed(x, delta, cmn)
        if stats is not None:
            r = (r - stats[0]) / stats[1]
        return l2(r)
    return emb

def build_models(E, k=None, rng=None):
    models = {}
    for sp in speakers:
        idx = np.arange(len(E[sp])) if k is None else (np.arange(k) if rng is None else rng.choice(len(E[sp]), k, replace=False))
        models[sp] = l2(np.mean([E[sp][i] for i in idx], axis=0))
    return models

def score_all(e, models):
    return {sp: cosine(e, m) for sp, m in models.items()}

def id_eval(emb, k=None, n_rep=1, seed=0):
    E = {sp: [emb(u["x"]) for u in SPK[sp]["enroll"]] for sp in speakers}
    T = [(sp, u["name"], emb(u["x"])) for sp in speakers for u in SPK[sp]["test"]]
    rng = np.random.default_rng(seed) if n_rep > 1 else None
    accs, margins, last = [], [], None
    for _ in range(n_rep):
        models = build_models(E, k, rng)
        res = []
        for sp, name, e in T:
            sc = score_all(e, models); pred = max(sc, key=sc.get)
            others = max(v for s_, v in sc.items() if s_ != sp)
            res.append(dict(file=name, true=sp, pred=pred, **{f"cos_{s_}": v for s_, v in sc.items()}, margin=sc[sp] - others))
        res = pd.DataFrame(res)
        accs.append((res.true == res.pred).mean()); margins.append(res.margin.mean()); last = (res, models)
    return dict(acc=float(np.mean(accs)), margin=float(np.mean(margins)), res=last[0], models=last[1], E=E)

emb_base = make_embedder(delta=True, cmn=False, z=False)
base = id_eval(emb_base)
print(f"Baseline (MFCC+Δ, không CMN): accuracy = {100*base['acc']:.1f}% trên {len(base['res'])} test utterance; "
      f"margin cosine TB (đúng − tốt nhất còn lại) = {base['margin']:.4f}")
sc_cols = [c for c in base["res"].columns if c.startswith("cos_")]
print("\nBảng score cosine của từng test utterance với mọi speaker model:")
print(base["res"].round(4).to_string(index=False))
base["res"].round(5).to_csv(RES / "4_4_speaker_id_scores.csv", index=False)

cm = confusion_matrix(base["res"].true, base["res"].pred, labels=speakers)
fig, ax = plt.subplots(figsize=(4.6, 4))
ConfusionMatrixDisplay(cm, display_labels=speakers).plot(cmap="Blues", ax=ax, colorbar=False)
ax.set_title("Speaker identification – clean test"); plt.tight_layout(); plt.savefig(FIG / "4_4_confusion_speaker.png", dpi=130); plt.show()

In [ ]:
# --- TN4: đặc trưng (MFCC vs MFCC+Δ), TN5: CMN on/off (+ z-score enrollment, mở rộng) ---
variants = {"MFCC-only": dict(delta=False, cmn=False), "MFCC+Δ": dict(delta=True, cmn=False),
            "MFCC+Δ + CMN": dict(delta=True, cmn=True), "MFCC-only + CMN": dict(delta=False, cmn=True),
            "MFCC+Δ + z-score enroll (mở rộng)": dict(delta=True, cmn=False, z=True)}
rows = []
for nm, kw in variants.items():
    r = id_eval(make_embedder(**kw))
    rows.append(dict(biến_thể=nm, accuracy_pct=round(100 * r["acc"], 1), margin_TB=round(r["margin"], 4),
                     cos_dung_TB=round(float(np.mean([row[f"cos_{row['true']}"] for _, row in r["res"].iterrows()])), 4)))
tn45 = pd.DataFrame(rows); tn45.to_csv(RES / "TN4_TN5_features_cmn.csv", index=False)
print(tn45.to_string(index=False))
print("\nmargin = cos(đúng) − cos(speaker thứ hai tốt nhất): càng lớn càng chắc chắn (accuracy có thể 100% nhưng margin nhỏ).")

# --- TN6: số enrollment utterance (trung bình 30 lần chọn ngẫu nhiên tập con) ---
rows = []
for k in CFG["token_grid"] + [None]:
    full = min(len(SPK[sp]["enroll"]) for sp in speakers)
    kk = None if (k is None or k >= full) else k
    r = id_eval(emb_base, k=kk, n_rep=30 if kk else 1, seed=CFG["seed"])
    rows.append(dict(enroll_tokens=("tất cả (%d)" % full) if kk is None else k, accuracy_pct=round(100 * r["acc"], 1), margin_TB=round(r["margin"], 4)))
tn6 = pd.DataFrame(rows).drop_duplicates(); tn6.to_csv(RES / "TN6_enroll_tokens.csv", index=False)
print("\nTN6 – số enrollment token:"); print(tn6.to_string(index=False))

**Nhận xét TN4–TN6 (điền sau khi xem bảng):** thêm Δ làm margin … ; CMN làm accuracy/margin … (CMN loại bỏ μ(c) – thành phần mang nhiều thông tin kênh *và* âm sắc người nói – nên trong baseline thống kê này có thể mất cue); tăng số enrollment từ 1 → … làm margin … và độ ổn định … (model trung bình ít nhiễu hơn). Không ép CMN phải tốt: báo cáo như kết quả ablation.

## 4.5 Speaker verification (FAR, FRR, threshold trên dev)

Claim "tôi là speaker $i$": $z=\cos(e,m_i)$, chấp nhận nếu $z\ge\theta$. **Dev** và **test** tách biệt:

- **Dev trials** tạo từ *enrollment* theo leave-one-out: mô hình speaker $i$ chỉ dùng các enrollment còn lại; utterance bị bỏ ra là genuine trial với model $i$ và impostor trial với model của mọi speaker $j\ne i$ (xây từ toàn bộ enrollment của $j$). Không dùng bất kỳ test utterance nào.
- $\theta$ chọn tại điểm FAR ≈ FRR trên dev (gần EER), **khoá lại**, rồi mới đo FAR/FRR trên **test trials** (test vs model đầy đủ). Không chỉnh $\theta$ sau khi xem test.

In [ ]:
def far_frr(gen, imp, th):
    gen, imp = np.asarray(gen), np.asarray(imp)
    return np.mean(imp[None, :] >= th[:, None], axis=1), np.mean(gen[None, :] < th[:, None], axis=1)   # FAR, FRR

E_all = base["E"]
dev_gen, dev_imp = [], []
for sp in speakers:
    for i, e in enumerate(E_all[sp]):
        rest = [E_all[sp][j] for j in range(len(E_all[sp])) if j != i]
        m_own = l2(np.mean(rest, axis=0))
        dev_gen.append(cosine(e, m_own))
        for sp2 in speakers:
            if sp2 != sp:
                dev_imp.append(cosine(e, base["models"][sp2]))
test_gen, test_imp = [], []
for sp in speakers:
    for u in SPK[sp]["test"]:
        e = emb_base(u["x"])
        for sp2 in speakers:
            (test_gen if sp2 == sp else test_imp).append(cosine(e, base["models"][sp2]))
print(f"Dev: {len(dev_gen)} genuine + {len(dev_imp)} impostor | Test: {len(test_gen)} genuine + {len(test_imp)} impostor")

lo = min(min(dev_gen), min(dev_imp), min(test_gen), min(test_imp)) - 0.01
hi = max(max(dev_gen), max(dev_imp), max(test_gen), max(test_imp)) + 0.01
TH = np.linspace(lo, hi, 4001)
dFAR, dFRR = far_frr(dev_gen, dev_imp, TH)
i_eer = int(np.argmin(np.abs(dFAR - dFRR) + 1e-6 * TH))             # nếu hoà: ưu tiên θ lớn (FAR nhỏ)
THETA = float(TH[i_eer])
print(f"θ chọn trên DEV (gần EER) = {THETA:.4f}  (FAR_dev = {dFAR[i_eer]:.3f}, FRR_dev = {dFRR[i_eer]:.3f})  -> KHOÁ θ")
tFAR, tFRR = far_frr(test_gen, test_imp, np.array([THETA]))
print(f"TEST tại θ đã khoá: FAR = {tFAR[0]:.3f} ({int(round(tFAR[0]*len(test_imp)))}/{len(test_imp)}), FRR = {tFRR[0]:.3f} ({int(round(tFRR[0]*len(test_gen)))}/{len(test_gen)})")
pd.DataFrame([dict(theta=THETA, FAR_dev=dFAR[i_eer], FRR_dev=dFRR[i_eer], FAR_test=tFAR[0], FRR_test=tFRR[0],
                   n_gen_test=len(test_gen), n_imp_test=len(test_imp))]).round(4).to_csv(RES / "4_5_verification.csv", index=False)

tgF, tgR = far_frr(test_gen, test_imp, TH)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(TH, dFAR, label="FAR (dev)", color="tab:red"); ax[0].plot(TH, dFRR, label="FRR (dev)", color="tab:blue")
ax[0].plot(TH, tgF, "--", label="FAR (test)", color="tab:red", alpha=0.5); ax[0].plot(TH, tgR, "--", label="FRR (test)", color="tab:blue", alpha=0.5)
ax[0].axvline(THETA, color="k", ls=":", label=f"θ dev = {THETA:.3f}"); ax[0].set_xlabel("θ"); ax[0].set_ylabel("tỷ lệ lỗi"); ax[0].legend(); ax[0].set_title("FAR/FRR theo θ")
ax[1].hist(dev_imp + test_imp, bins=20, alpha=0.6, label="impostor", color="tab:red"); ax[1].hist(dev_gen + test_gen, bins=20, alpha=0.6, label="genuine", color="tab:green")
ax[1].axvline(THETA, color="k", ls=":"); ax[1].set_xlabel("cosine score"); ax[1].legend(); ax[1].set_title("Phân bố score genuine vs impostor")
plt.tight_layout(); plt.savefig(FIG / "4_5_far_frr.png", dpi=130); plt.show()

**Nhận xét:** θ phải chọn trên dev vì nếu chọn trên test, ta đang "luyện" ngưỡng vào chính dữ liệu dùng để báo cáo → FAR/FRR đẹp giả tạo (lạc quan). Khi θ tăng: FAR giảm (khó nhận nhầm kẻ giả mạo) nhưng FRR tăng (từ chối nhầm chính chủ) – xem hai đường cắt nhau ở gần EER. Kích thước tập trial ở đây nhỏ nên FAR/FRR chỉ là ước lượng thô (mỗi lỗi = 1/N).

## 4.8a Tích hợp: speaker-ID trên noisy và enhanced speech (TN8a, câu 12)

Model người nói **giữ nguyên** (enroll trên giọng sạch). Test utterance được trộn nhiễu rồi (a) để nguyên, (b) qua spectral subtraction, (c) qua Wiener. Đo accuracy, margin, và FAR/FRR tại θ đã khoá (verification).

In [ ]:
def eval_conditions(noise_name, R):
    v = NOISES[noise_name]
    out = {k: dict(pred=[], margin=[], true_score=[], gen=[], imp=[]) for k in ["clean", "noisy", "SS", "Wiener"]}
    for sp in speakers:
        for u in SPK[sp]["test"]:
            s = u["x"]; y, alpha, vv = add_noise_at_snr(s, v, R); Pn = noise_psd(alpha * vv)
            sigs = {"clean": s, "noisy": y, "SS": enhance(y, Pn, "ss"), "Wiener": enhance(y, Pn, "wiener")}
            for k, sig in sigs.items():
                sc = score_all(emb_base(sig), base["models"])
                out[k]["pred"].append(max(sc, key=sc.get) == sp)
                out[k]["margin"].append(sc[sp] - max(x for s2, x in sc.items() if s2 != sp))
                out[k]["true_score"].append(sc[sp])
                for sp2, val in sc.items():
                    out[k]["gen" if sp2 == sp else "imp"].append(val)
    return out

rows = []
for nz in NOISES:
    for R in CFG["snrs"]:
        o = eval_conditions(nz, R)
        for k in ["clean", "noisy", "SS", "Wiener"]:
            far_, frr_ = far_frr(o[k]["gen"], o[k]["imp"], np.array([THETA]))
            rows.append(dict(noise=nz, SNR=R, điều_kiện=k, accuracy_pct=round(100 * np.mean(o[k]["pred"]), 1),
                             margin_TB=round(float(np.mean(o[k]["margin"])), 4), cos_dung_TB=round(float(np.mean(o[k]["true_score"])), 4),
                             FAR_tai_theta=round(float(far_[0]), 3), FRR_tai_theta=round(float(frr_[0]), 3)))
tn8a = pd.DataFrame(rows); tn8a.to_csv(RES / "TN8a_id_noisy_enhanced.csv", index=False)
print(f"Speaker-ID / verification (θ = {THETA:.4f} khoá từ dev) với model enroll trên giọng sạch:")
print(tn8a.to_string(index=False))

## 4.6 Tạo hỗn hợp hai nguồn (2 nguồn – 2 microphone)

Chọn hai nguồn **khác người nói**, chuẩn hoá RMS, cắt cùng độ dài; trộn tức thời $x=As$ với $A$ khả nghịch, ví dụ $A=\begin{bmatrix}1&0.6\\0.4&1\end{bmatrix}$. $A$ chỉ dùng để *tạo dữ liệu có ground truth*, **không** đưa vào thuật toán ICA.

Nếu chưa có `separation/source_A.wav` và `source_B.wav`, notebook tạm nối các utterance của hai người nói khác nhau trong `clean/` (ghi rõ nguồn dữ liệu bên dưới).

In [ ]:
def rms_normalize(x):
    return x / (rms(x) + 1e-12)

pa, pb = SEP / "source_A.wav", SEP / "source_B.wav"
if pa.exists() and pb.exists():
    s1, _ = load16k(pa); s2, _ = load16k(pb); SRC_NOTE = "lab5_data/separation/source_A.wav + source_B.wav"
else:
    cat = lambda sp: np.concatenate([u["x"] for u in (SPK[sp]["enroll"] + SPK[sp]["test"])][:4])
    s1, s2 = cat(speakers[0]), cat(speakers[1]); SRC_NOTE = f"dự phòng: nối utterance của {speakers[0]} và {speakers[1]}"
L = min(len(s1), len(s2))
S = np.vstack([rms_normalize(s1[:L]), rms_normalize(s2[:L])])
print("Nguồn:", SRC_NOTE, f"| độ dài {L/FS:.1f}s | tương quan nguồn ρ(s1,s2) = {np.corrcoef(S[0], S[1])[0,1]:+.3f} (gần 0 = gần độc lập)")

rows = []
for nm, Al in CFG["A_list"].items():
    A = np.array(Al)
    rows.append(dict(A=nm, ma_trận=str(Al), det=round(float(np.linalg.det(A)), 4), cond=round(float(np.linalg.cond(A)), 1)))
print(pd.DataFrame(rows).to_string(index=False))

A0 = np.array(CFG["A_list"]["dễ"])
X0 = A0 @ S
save_wav(AUD / "mix_mic1.wav", X0[0]); save_wav(AUD / "mix_mic2.wav", X0[1])
save_wav(AUD / "source_1_truth.wav", S[0]); save_wav(AUD / "source_2_truth.wav", S[1])
print(f"\nA = {A0.tolist()}  det(A) = {np.linalg.det(A0):.3f}, cond(A) = {np.linalg.cond(A0):.2f}")
print("Hai hàng của A gần tỉ lệ nhau ⇒ det(A) → 0 ⇒ hai mixture gần như giống nhau (cùng hướng) ⇒ A⁻¹ khuếch đại nhiễu "
      "và thông tin phân biệt hai nguồn rất nhỏ ⇒ tách khó.")

## 4.7 FastICA / BSS

`FastICA(n_components=2, whiten='unit-variance')` trên ma trận $[x_1,x_2]$. Hai bất định của ICA: **permutation** (không biết output nào ứng với nguồn nào) và **scale/sign** (biên độ, dấu bất kỳ). Vì vậy ghép estimate với source bằng $|\rho|$ (thuật toán Hungarian trên ma trận $|\rho|$), đảo dấu nếu tương quan âm, chuẩn hoá mức rồi mới so sánh. Ngoài $|\rho|$ còn báo **SDR** (dB) = tỷ số năng lượng giữa phần estimate chiếu lên nguồn thật và phần dư.

In [ ]:
def sdr_db(s, e):
    a = np.dot(e, s) / np.dot(s, s); proj = a * s
    return float(10 * np.log10(np.sum(proj ** 2) / (np.sum((e - proj) ** 2) + 1e-12)))

def corr_matrix(S_, E_):
    return np.array([[abs(np.corrcoef(S_[i], E_[j])[0, 1]) for j in range(2)] for i in range(2)])

def ica_separate(X):
    ica = FastICA(n_components=2, whiten="unit-variance", random_state=CFG["seed"], max_iter=2000, tol=1e-4)
    Sh = ica.fit_transform(X.T).T                                  # 2 × L
    C = corr_matrix(S, Sh)                                         # TRƯỚC khi ghép
    r, c = linear_sum_assignment(-C)                               # ghép permutation theo |ρ|
    Sm = np.zeros_like(Sh)
    for i, j in zip(r, c):
        sg = np.sign(np.corrcoef(S[i], Sh[j])[0, 1]) or 1.0        # sửa dấu
        Sm[i] = sg * Sh[j]
    Sm = np.vstack([rms_normalize(x) for x in Sm])
    return Sh, Sm, C, corr_matrix(S, Sm), [sdr_db(S[i], Sm[i]) for i in range(2)], list(c)

Sh0, Sm0, C0, Cm0, sdr0, perm0 = ica_separate(X0)
print("|ρ(source_i, estimate_j)| TRƯỚC khi ghép (hàng = source, cột = output ICA):\n", np.round(C0, 3))
print("ghép: source 1 ← output", perm0[0], ", source 2 ← output", perm0[1])
print("|ρ| SAU khi ghép (đã đổi thứ tự + dấu):\n", np.round(Cm0, 3))
print("SDR sau ghép (dB):", np.round(sdr0, 2))
save_wav(AUD / "separated_1.wav", Sm0[0]); save_wav(AUD / "separated_2.wav", Sm0[1])

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
for a, (M_, nm) in zip(ax, [(C0, "TRƯỚC khi ghép"), (Cm0, "SAU khi ghép")]):
    im = a.imshow(M_, vmin=0, vmax=1, cmap="viridis")
    a.set_xticks([0, 1]); a.set_yticks([0, 1]); a.set_xlabel("estimate j"); a.set_ylabel("source i"); a.set_title(f"|ρ| {nm}")
    for i in range(2):
        for j in range(2):
            a.text(j, i, f"{M_[i, j]:.2f}", ha="center", va="center", color="w" if M_[i, j] < 0.6 else "k")
plt.tight_layout(); plt.savefig(FIG / "4_7_ica_correlation.png", dpi=130); plt.show()

fig, ax = plt.subplots(3, 2, figsize=(13, 7), sharex=True)
tt = np.arange(L) / FS
for r_, (sig, nm) in enumerate([(S, "nguồn thật"), (X0, "mixture tại 2 micro"), (Sm0, "ICA (đã ghép + chuẩn hoá)")]):
    for c_ in range(2):
        ax[r_, c_].plot(tt, sig[c_], lw=0.5, color="k"); ax[r_, c_].set_title(f"{nm} – kênh {c_+1}")
ax[-1, 0].set_xlabel("giây"); ax[-1, 1].set_xlabel("giây")
plt.tight_layout(); plt.savefig(FIG / "4_7_ica_waveforms.png", dpi=130); plt.show()

In [ ]:
# --- TN7: mixing matrix dễ / trung bình / gần suy biến, trong nhiều điều kiện nhiễu ---
rng = np.random.default_rng(CFG["seed"])
def add_mix_noise(X, kind, snr):
    Xn = X.copy()
    for k in range(2):
        if kind == "trắng":
            n = rng.standard_normal(X.shape[1])
        else:
            v = NOISES[CASE_NOISE]; off = (k * 7919) % max(len(v) - X.shape[1], 1)
            n = np.tile(v, int(np.ceil(X.shape[1] / len(v)) + 1))[off:off + X.shape[1]]
        n = n * np.sqrt(power(X[k]) / (power(n) * 10 ** (snr / 10)))     # SNR tính theo công suất của từng kênh mixture
        Xn[k] = X[k] + n
    return Xn

def quant16(X):
    g = 0.95 / np.max(np.abs(X)); return np.round(X * g * 32767) / 32767 / g

TN7_CONDS = {"sạch (float64)": lambda X: X, "lượng tử 16-bit": quant16,
             "+ nhiễu trắng 40 dB": lambda X: add_mix_noise(X, "trắng", 40),
             "+ nhiễu trắng 30 dB": lambda X: add_mix_noise(X, "trắng", 30),
             "+ nhiễu trắng 20 dB": lambda X: add_mix_noise(X, "trắng", 20)}
rows = []
for nm, Al in CFG["A_list"].items():
    A = np.array(Al)
    for cn, fn in TN7_CONDS.items():
        _, Sm_, C_, Cm_, sd_, _ = ica_separate(fn(A @ S))
        rows.append(dict(A=nm, det=round(float(np.linalg.det(A)), 4), cond=round(float(np.linalg.cond(A)), 1), điều_kiện=cn,
                         rho_TB=round(float(np.mean(np.diag(Cm_))), 3), SDR_TB_dB=round(float(np.mean(sd_)), 2)))
tn7 = pd.DataFrame(rows); tn7.to_csv(RES / "TN7_mixing_matrix.csv", index=False)
print("TN7 – ảnh hưởng của det(A) lên chất lượng tách (sau ghép), theo điều kiện mixture:")
print(tn7.to_string(index=False))
print("\nSDR TB (dB) – hàng: A, cột: điều kiện:"); print(tn7.pivot_table(index=["A", "det"], columns="điều_kiện", values="SDR_TB_dB", sort=False).round(2).to_string())

**Nhận xét TN7:** với mixture *sạch* và đủ chính xác số, ICA (sau whitening) có tính chất **equivariance** – chất lượng tách gần như **không phụ thuộc** vào $A$, nên cột "sạch" cho kết quả gần giống nhau ở mọi $A$ (đó là kết quả đúng, không phải lỗi). Ảnh hưởng của det(A) xuất hiện khi mixture có nhiễu/sai số: với $A$ gần suy biến, hai mixture gần như cùng một tổ hợp của hai nguồn, phần *khác biệt* giữa chúng (thứ ICA thực sự dùng để tách) rất nhỏ so với mức nhiễu, nên cùng một mức nhiễu gây suy giảm |ρ| và SDR lớn hơn so với $A$ dễ. Điền số liệu từ bảng: SDR ở nhiễu 30 dB giảm từ … (A dễ) xuống … (A gần suy biến).

## 4.8b Tích hợp: ICA khi mixture có nhiễu, và giới hạn của mô hình trộn tức thời (TN8b, câu 11)

- **Nhiễu cộng** thêm vào hai mixture (trắng, và nhiễu thật nếu có) ở SNR 30/20/10 dB: mô hình $x=As$ không còn đúng (có thêm nguồn thứ ba/nhiễu), ICA 2×2 suy giảm.
- **Reverberation (convolutive mixing):** mỗi đường truyền là một đáp ứng xung ngắn (nhiều tia phản xạ suy giảm theo hàm mũ) thay vì một hệ số → mô hình tức thời sai về bản chất.

In [ ]:
rows = []
_, _, _, Cm_c, sd_c, _ = ica_separate(X0)
rows.append(dict(điều_kiện="mixture sạch (A dễ)", rho_TB=round(float(np.mean(np.diag(Cm_c))), 3), SDR_TB_dB=round(float(np.mean(sd_c)), 2)))
for kind in ["trắng", CASE_NOISE]:
    for snr in CFG["ica_noise_snrs"]:
        _, _, _, Cm_, sd_, _ = ica_separate(add_mix_noise(X0, kind, snr))
        rows.append(dict(điều_kiện=f"+ nhiễu {kind} {snr} dB", rho_TB=round(float(np.mean(np.diag(Cm_))), 3), SDR_TB_dB=round(float(np.mean(sd_)), 2)))

# trộn convolutive: h_ij = hệ số A_ij * (xung trực tiếp + các tia phản xạ suy giảm trong 20 ms)
def reverb_ir(strength, length_ms=20, n_echo=8):
    h = np.zeros(int(FS * length_ms / 1000)); h[0] = 1.0
    for _ in range(n_echo):
        d = int(rng.integers(8, len(h))); h[d] += strength * np.exp(-3.0 * d / len(h)) * rng.choice([-1, 1])
    return h
from scipy.signal import fftconvolve
for st in [0.3, 0.6, 1.0]:
    Xc = np.vstack([fftconvolve(S[0], A0[0, 0] * reverb_ir(st))[:L] + fftconvolve(S[1], A0[0, 1] * reverb_ir(st))[:L],
                    fftconvolve(S[0], A0[1, 0] * reverb_ir(st))[:L] + fftconvolve(S[1], A0[1, 1] * reverb_ir(st))[:L]])
    _, _, _, Cm_, sd_, _ = ica_separate(Xc)
    rows.append(dict(điều_kiện=f"reverberation (cường độ phản xạ {st})", rho_TB=round(float(np.mean(np.diag(Cm_))), 3), SDR_TB_dB=round(float(np.mean(sd_)), 2)))
tn8b = pd.DataFrame(rows); tn8b.to_csv(RES / "TN8b_ica_noise_reverb.csv", index=False)
print(tn8b.to_string(index=False))

## 5. Case study tổng hợp – "Trợ lý phòng họp mini"

Chạy tuần tự ba chức năng trên cùng dữ liệu: (1) làm sạch câu nói bị nhiễu quạt; (2) đoán người nói; (3) tách hai người nói chồng lấn.

In [ ]:
R = CFG["case_snr"]; sp_true = [sp for sp in speakers if any(u["name"] == case_s["name"] for u in SPK[sp]["test"])][0]
y_c, a_c, vv_c = add_noise_at_snr(case_s["x"], v_case, R); Pn_c = noise_psd(a_c * vv_c)
sigs = {"clean": case_s["x"], "noisy": y_c, "SS": enhance(y_c, Pn_c, "ss"), "Wiener": enhance(y_c, Pn_c, "wiener")}
rows = []
for k, sig in sigs.items():
    sc = score_all(emb_base(sig), base["models"]); order = sorted(sc, key=sc.get, reverse=True)
    rows.append(dict(tín_hiệu=k, SNR_dB=("-" if k == "clean" else round(snr_aligned(case_s["x"], sig)[0], 2)),
                     dự_đoán=order[0], đúng=order[0] == sp_true, cos_đúng=round(sc[sp_true], 4),
                     margin_so_với_thứ_2=round(sc[sp_true] - max(v for s_, v in sc.items() if s_ != sp_true), 4),
                     **{f"cos_{s_}": round(v, 4) for s_, v in sc.items()}))
case_final = pd.DataFrame(rows); case_final.to_csv(RES / "5_case_study.csv", index=False)
print(f"(1)+(2) Câu {case_s['name']} (người nói thật: {sp_true}) + nhiễu {CASE_NOISE} @ {R} dB:"); print(case_final.to_string(index=False))
print(f"\n(3) Tách 2 nguồn (A dễ): |ρ| sau ghép = {np.round(np.diag(Cm0), 3)}, SDR = {np.round(sdr0, 2)} dB")
print("\nFile âm thanh:", sorted(p.name for p in AUD.glob("*.wav") if "_" in p.name and not any(t in p.name for t in ["beta", "gamma"]))[:20])

## Trả lời câu hỏi báo cáo (mục 7)

**1. Vì sao spectral subtraction chỉ sửa magnitude, giữ phase noisy? Hạn chế khi SNR rất thấp?** Phổ công suất nhiễu ước lượng được dễ (là kỳ vọng $E|V|^2$), còn pha của nhiễu là ngẫu nhiên không ước lượng được cho từng ô; tai người nhạy với magnitude hơn pha (khi SNR ≳ 6–8 dB sai pha khó nghe thấy) nên mượn pha của $Y$ là xấp xỉ hợp lý. Khi SNR rất thấp, pha noisy chủ yếu là pha của nhiễu → speech tái tạo có pha sai: giọng "thô/khàn", tính ồn còn lại và giới hạn trên của chất lượng không nâng lên được dù magnitude ước lượng tốt.

**2. Vai trò β, γ; vì sao tăng β giảm nhiễu nhưng tăng méo?** β ≥ 1 là hệ số *over-subtraction* bù cho sai số ước lượng PSD nhiễu (trừ nhiều hơn kỳ vọng để dập các đỉnh nhiễu ngẫu nhiên); γ là *spectral floor* $\gamma\hat P_v$ ngăn giá trị âm/lỗ phổ sâu. Tăng β làm trừ cả một phần năng lượng speech yếu (phụ âm, vùng cao tần) → speech distortion (SD_dB giảm), và các ô bị chặn đến floor xen kẽ các ô còn năng lượng → gai phổ cô lập (musical noise, KR tăng). Floor lớn che bớt gai nhưng để lại nền nhiễu (xem TN2, TN3).

**3. Ý nghĩa Wiener $H=P_s/(P_s+P_v)$ ở vùng speech-dominant và noise-dominant?** Vùng speech-dominant ($P_s\gg P_v$): $H\to1$, giữ nguyên tín hiệu. Vùng noise-dominant ($P_v\gg P_s$): $H\to0$, dập mạnh. Ở giữa là suy giảm **mượt** tỷ lệ với SNR cục bộ, nên ít gai hơn so với phép trừ cứng có floor.

**4. Vì sao SNR_out phải tính sau khi căn chỉnh? Nếu có trễ 10 ms thì sai theo hướng nào?** SNR so sánh $s[n]$ với $\hat s[n]$ từng mẫu; nếu $\hat s$ bị trễ, sai số $s-\hat s$ gồm cả phần chênh pha do trễ (với tín hiệu hữu thanh tuần hoàn, trễ 10 ms = 160 mẫu có thể gần như ngược pha ở nhiều thành phần) → năng lượng sai số lớn hơn thực chất → **SNR_out bị đánh giá THẤP hơn** thực tế (thuật toán trông tệ hơn), thậm chí âm. Notebook dùng `snr_aligned` (tìm lag ±30 ms bằng tương quan chéo) và in lag, ở đây lag = 0 vì STFT/ISTFT với `center=True` không gây trễ.

**5. Speaker identification vs verification (output và metric)?** Identification: bài toán phân loại *đóng* – chọn 1 trong N người đã enroll; đầu ra là nhãn; metric accuracy/confusion matrix. Verification: kiểm tra claim "tôi là $i$" – đầu ra nhị phân accept/reject theo ngưỡng θ; metric FAR, FRR (và EER, DET) – phụ thuộc θ.

**6. Vì sao θ không chọn trên test? Trade-off FAR–FRR?** Chọn θ trên test là "học" ngưỡng từ dữ liệu báo cáo nên kết quả lạc quan, không phản ánh hiệu năng trên dữ liệu mới; phải chọn trên dev rồi khoá. θ tăng → khó chấp nhận → FAR giảm, FRR tăng; θ giảm thì ngược lại; EER là điểm hai đường cắt nhau. Ứng dụng bảo mật chọn θ cao (ưu tiên FAR thấp), ứng dụng tiện dụng chọn θ thấp hơn.

**7. Vì sao MFCC (thiết kế cho nội dung âm học) vẫn chứa thông tin người nói?** MFCC mô tả đường bao phổ ngắn hạn, mà đường bao phổ phụ thuộc hình dạng/chiều dài đường dẫn thanh của từng người (formant, độ nghiêng phổ) cùng đặc tính nguồn glottal. Thống kê μ, σ theo thời gian trung bình hoá phần phụ thuộc nội dung, giữ lại phần ổn định theo người nói.

**8. Vì sao CMN vừa hữu ích cho channel robustness vừa bất lợi cho baseline speaker-ID?** Kênh/micro nhân vào phổ ⇒ cộng hằng số vào cepstrum, nên trừ trung bình theo utterance loại được ảnh hưởng này (hữu ích khi enroll và test khác micro/phòng). Nhưng trung bình cepstrum cũng chứa **đường bao phổ trung bình của giọng người nói** – chính là cue mạnh nhất của baseline $\mu(c)$ – nên CMN làm mất cue đó (xem TN5).

**9. Vì sao ICA không biết trước thứ tự và biên độ tuyệt đối?** Nếu $x=As$ thì với mọi ma trận hoán vị $P$ và ma trận đường chéo $D$ khả nghịch, $x=(AD^{-1}P^{-1})(PDs)$ cũng là một mô hình hợp lệ với các nguồn độc lập $PDs$; ICA chỉ dựa vào tính độc lập/phi-Gauss nên không phân biệt được các nghiệm này ⇒ bất định hoán vị và scale/dấu.

**10. Nếu hai hàng của A gần tỉ lệ nhau?** $\det(A)\to0$, A gần suy biến, $A^{-1}$ không tồn tại/ill-conditioned: hai mixture gần như cùng một tín hiệu nên không thể khôi phục hai nguồn riêng; nhiễu nhỏ cũng bị khuếch đại rất mạnh khi nghịch đảo. Về lý thuyết ICA sau whitening là bất biến với $A$ khi mixture sạch (xem cột "sạch" ở TN7), nhưng khi có nhiễu/sai số, phần khác biệt giữa hai mixture nhỏ so với nhiễu nên |ρ| và SDR suy giảm theo det(A) (xem TN7).

**11. Instantaneous mixing khác phòng có reverberation ở điểm nào?** Instantaneous: $x_i[n]=\sum_j a_{ij}s_j[n]$ – mỗi nguồn chỉ nhân hệ số, không trễ. Thực tế: $x_i=\sum_j h_{ij}*s_j$ với $h_{ij}$ là đáp ứng xung phòng dài hàng chục–trăm ms (trễ lan truyền + phản xạ). Mô hình thành convolutive: cần bộ tách là bộ lọc FIR/ICA trong miền tần số cho từng bin (và xử lý hoán vị theo từng bin). ICA 2×2 cơ bản suy giảm (xem TN8b).

**12. Enhancement có luôn cải thiện speaker recognition không?** Không đảm bảo. Xem bảng TN8a và ô phân tích bên dưới (số liệu theo từng noise/SNR): spectral subtraction/Wiener loại bớt nhiễu nên đưa phân bố MFCC về gần giọng sạch hơn, nhưng cũng để lại méo/musical noise và làm biến đổi thống kê σ(c), Δ – có thể giúp ở SNR thấp–trung bình và không giúp (thậm chí hại) ở SNR cao.

In [ ]:
# Phân tích số liệu cho câu 12
d = tn8a.pivot_table(index=["noise", "SNR"], columns="điều_kiện", values="accuracy_pct")
m = tn8a.pivot_table(index=["noise", "SNR"], columns="điều_kiện", values="margin_TB")
print("Accuracy (%) speaker-ID:"); print(d[["clean", "noisy", "SS", "Wiener"]].to_string())
print("\nMargin cosine TB:"); print(m[["clean", "noisy", "SS", "Wiener"]].round(4).to_string())
for meth in ["SS", "Wiener"]:
    up, eq, dn = (d[meth] > d["noisy"]).sum(), (d[meth] == d["noisy"]).sum(), (d[meth] < d["noisy"]).sum()
    mu, md = (m[meth] > m["noisy"]).sum(), (m[meth] < m["noisy"]).sum()
    print(f"\n{meth}: accuracy so với noisy – tăng {up}, bằng {eq}, giảm {dn} (trên {len(d)} điều kiện); margin tăng {mu}, giảm {md}.")

In [ ]:
print("=" * 64); print("TÓM TẮT KẾT QUẢ LAB 5 – 3 KẾT LUẬN KỸ THUẬT CÓ SỐ LIỆU"); print("=" * 64)
# 1) enhancement
pv = tn1.pivot_table(index="SNR_in", columns="method", values="dSNR")
print(f"[1] Enhancement (TB trên nhiễu {list(NOISES)}): ΔSNR SS = {pv['SS'].round(2).to_dict()}, Wiener = {pv['Wiener'].round(2).to_dict()} dB. "
      f"Wiener {'cao hơn' if (pv['Wiener'] > pv['SS']).all() else 'không luôn cao hơn'} SS ở "
      f"{int((pv['Wiener'] > pv['SS']).sum())}/{len(pv)} mức SNR.")
b = tn2.set_index("beta")
print(f"    β {b.index.min()}→{b.index.max()}: giảm nhiễu NR {b.NR_dB.iloc[0]:.1f}→{b.NR_dB.iloc[-1]:.1f} dB, "
      f"SD (độ giữ speech) {b.SD_dB.iloc[0]:.1f}→{b.SD_dB.iloc[-1]:.1f} dB, KR musical {b.KR_musical.iloc[0]:.2f}→{b.KR_musical.iloc[-1]:.2f}.")
# 2) speaker
print(f"[2] Speaker-ID clean: {100*base['acc']:.1f}% (margin {base['margin']:.4f}); ablation: "
      + "; ".join(f"{r.biến_thể}: {r.accuracy_pct}%/{r.margin_TB}" for r in tn45.itertuples()))
print(f"    Verification: θ={THETA:.4f} (chọn trên dev), test FAR={tFAR[0]:.3f}, FRR={tFRR[0]:.3f}.")
dd = tn8a.pivot_table(index=["noise", "SNR"], columns="điều_kiện", values="accuracy_pct")
print(f"    Enhancement vs noisy (accuracy): SS tăng {(dd['SS'] > dd['noisy']).sum()}/giảm {(dd['SS'] < dd['noisy']).sum()}, "
      f"Wiener tăng {(dd['Wiener'] > dd['noisy']).sum()}/giảm {(dd['Wiener'] < dd['noisy']).sum()} trên {len(dd)} điều kiện.")
# 3) ICA
print("[3] ICA – SDR (dB) theo A: " + "; ".join(
    f"A {a} (det={d}): sạch {g.set_index('điều_kiện').SDR_TB_dB['sạch (float64)']}, +nhiễu 30 dB {g.set_index('điều_kiện').SDR_TB_dB['+ nhiễu trắng 30 dB']}"
    for (a, d), g in tn7.groupby(["A", "det"], sort=False)))
print("    " + "; ".join(f"{r.điều_kiện}: |ρ|={r.rho_TB}, SDR={r.SDR_TB_dB} dB" for r in tn8b.itertuples()))